# Importaciones

In [1]:
include("dependencies.jl")
include("helpers.jl")
include("wrappers.jl")
# Cargamos los datos preparados en el notebook anterior al instante
JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval n_features

6-element Vector{Symbol}:
 :df_trainval
 :df_test
 :X_trainval
 :y_trainval
 :folds_trainval
 :n_features

# Modelos básicos y selección de atributos (20%)

In [2]:
# Definición de diccionarios de configuración
dic_filtros = Dict(
    "ANOVA" => MyANOVAFilter(n_features=n_features),
    "Pearson" => MyPearsonFilter(n_features=n_features),
    "Spearman" => MySpearmanFilter(n_features=n_features),
    "Kendall" => MyKendallFilter(n_features=n_features),
    "MI" => MyMIFilter(n_features=n_features),
    "RFE" => MyRFEFilter(n_features=n_features)
)

dic_reducciones = Dict(
    "Sin reducción" => IdentityTransformer(),
    "PCA" => PCA(variance_ratio=0.95),
    "ICA" => ICA(outdim=2, maxiter=10000,tol=0.5),
    "LDA" => LDA(method=:whiten, outdim=5) 
)

dic_modelos = Dict(
    "NeuralNetwork_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
    "NeuralNetwork_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
    "NeuralNetwork_100_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50))),
    
    "KNN_1" => KNNClassifier(K=1),
    "KNN_10" => KNNClassifier(K=10),
    "KNN_20" => KNNClassifier(K=20),

    "SVM_0.1" => ProbabilisticSVC(cost=0.1),
    "SVM_0.5" => ProbabilisticSVC(cost=0.5),
    "SVM_1.0" => ProbabilisticSVC(cost=1.0)
);

In [5]:
function run_experiment(dic_filtros, dic_reducciones, dic_modelos, output_file; 
                              X=X_trainval, y=y_trainval, folds=folds_trainval)
    
    # Comprobamos si ya existe un archivo de resultados previo
    if isfile(output_file)
        println(">>> Archivo de checkpoint encontrado: $output_file")
        
        # Leemos el archivo existente
        results_df = CSV.read(output_file, DataFrame)
        
        # Creamos un conjunto con las combinaciones ya hechas para búsqueda rápida
        combinaciones_hechas = Set([
            (r.Filter, r.Reduction, r.Model) 
            for r in eachrow(results_df)
        ])
        
        println(">>> Se han detectado $(length(combinaciones_hechas)) experimentos ya completados. Se saltarán.")
    else
        println(">>> Iniciando experimento desde cero.")
        # Si no existe, inicializamos el DataFrame vacío como siempre
        results_df = DataFrame(
            Filter = String[], Reduction = String[], Model = String[],
            Accuracy = Any[], Accuracy_Mean = Float64[], F1_Score = Float64[], Recall = Float64[]
        )
        combinaciones_hechas = Set{Tuple{String, String, String}}()
    end
    
    # Métricas a evaluar
    measures = [accuracy, multiclass_f1score, recall]

    # Bucle de ejecución
    for (filt_name, filt_model) in dic_filtros
        for (red_name, red_model) in dic_reducciones
            for (mod_name, mod_model) in dic_modelos
                
                # Si esta combinación ya está en el set de hechas, saltamos a la siguiente
                if (filt_name, red_name, mod_name) in combinaciones_hechas
                    continue 
                end

                println(">>> Evaluando: $filt_name + $red_name + $mod_name")
                
                # Instanciamos PersonalizedPipeline
                pipe = PersonalizedPipeline(
                    scaler    = MyMinMaxScaler(), 
                    filter    = filt_model,      
                    reduction = red_model,       
                    clf       = mod_model        
                )
                
                try
                    # Evaluación con Cross-Validation
                    evaluation = evaluate!(
                        pipe, X, y,
                        resampling = folds, measures = measures, verbosity = 0,
                        acceleration = CPUThreads()
                    )
                    
                    # Extraer métricas
                    acc_per_fold = evaluation.measurement[1]
                    f1 = evaluation.measurement[2]
                    recall = evaluation.measurement[3]
                    acc_mean = mean(acc_per_fold)
                    
                    println("    Accuracy media: $acc_mean | F1: $f1")
                    
                    # Guardar en DataFrame
                    push!(results_df, (filt_name, red_name, mod_name, string(acc_per_fold), acc_mean, f1, recall))
                    
                    # Guardado continuo en CSV
                    CSV.write(output_file, results_df)
                    
                catch e
                    println("!!! Error en $filt_name + $red_name + $mod_name: $e")
                    
                    # Registramos el fallo también para que no se atasque intentándolo siempre
                    push!(results_df, (filt_name, red_name, mod_name, "ERROR", NaN, NaN, NaN))
                    CSV.write(output_file, results_df)
                end
            end
        end
    end
    
    println("Experimento finalizado.")
    return results_df
end

run_experiment (generic function with 1 method)

In [6]:
# Ejecutar y guardar
df_resultados_basicos = run_experiment(
    dic_filtros, 
    dic_reducciones, 
    dic_modelos, 
    "resultados_basicos.csv"
)

>>> Archivo de checkpoint encontrado: resultados_basicos.csv


Excessive output truncated after 10485921 bytes.

>>> Se han detectado 17 experimentos ya completados. Se saltarán.
>>> Evaluando: MI + PCA + SVM_0.5

MethodError: MethodError: Cannot `convert` an object of type String to an object of type Float64
The function `convert` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  convert(::Type{T}, !Matched::Flux.NilNumber.Nil) where T<:Number
   @ Flux C:\Users\Pc\.julia\packages\Flux\WMUyh\src\outputsize.jl:20
  convert(::Type{T}, !Matched::Union{InitialValues.SpecificInitialValue{typeof(*)}, InitialValues.SpecificInitialValue{typeof(Base.mul_prod)}}) where T<:Union{AbstractString, Number}
   @ InitialValues C:\Users\Pc\.julia\packages\InitialValues\OWP8V\src\InitialValues.jl:258
  convert(::Type{S}, !Matched::CategoricalArrays.CategoricalValue) where S<:Union{AbstractChar, AbstractString, Number}
   @ CategoricalArrays C:\Users\Pc\.julia\packages\CategoricalArrays\ptzwC\src\value.jl:92
  ...
